# 06 — Experience GAM diligence: what the real data said

**This notebook needs no licensed data.** It reads the *findings* committed under
`docs/measurements/` — aggregate reports generated verbatim by
`scripts/experience_diligence.py`. The HMD and SOA-ILEC source files never enter
the repo (Design Anchor 6), so anyone who clones this repository can re-run
everything below.

It is the visual companion to `docs/MEASUREMENT_experience_gam_hmd.md` and
`docs/MEASUREMENT_experience_gam_ilec.md`, and the canvas for quantifying effects
across models, populations and features.

Every quantitative claim in the measurement documents is re-derived here from the
JSON and asserted, so **executing this notebook end to end is a check on those
documents** — if a number in the prose drifts from the committed report, a cell
below fails.

In [ ]:
from pathlib import Path
import json

import numpy as np
import polars as pl

try:  # notebooks/ when run as a file, repo root when run by the test harness
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd()
    if not (ROOT / "docs" / "measurements").is_dir():
        ROOT = ROOT.parent

MEASUREMENTS = ROOT / "docs" / "measurements"


def load(name: str) -> dict:
    with (MEASUREMENTS / f"{name}.json").open() as handle:
        return json.load(handle)


REPORTS = {
    "HMD USA": load("experience_gam_hmd_usa"),
    "HMD E&W": load("experience_gam_hmd_gbrtenw"),
    "ILEC pooled": load("experience_gam_ilec"),
    "ILEC banded": load("experience_gam_ilec_duration_banded"),
}
for name, report in REPORTS.items():
    assert report["schema_version"] == 1, name
# Sections that could not run because a committed report predates the
# feature they read. Tracked rather than merely printed: the notebook's
# contract is that executing it checks the prose, and a section that
# silently prints instead of asserting would overstate that coverage.
# tests/test_notebooks/ pins this list, so a new gap fails and a closed
# one fails too — either way it cannot drift unnoticed.
DEGRADED: list[str] = []

print(f"loaded {len(REPORTS)} reports, schema v1")

## 1. Fit quality across the four runs

`dispersion` is the Pearson φ. Poisson assumes φ = 1; anything far above it means
unmodelled heterogeneity, and the delta-method bands must be scaled by √φ or they
overstate precision. The `band_inflation` column is that scaling as applied.

In [ ]:
rows = []
for name, r in REPORTS.items():
    fit, agg = r["fit"], r["aggregation"]
    rows.append({
        "run": name,
        "cells": agg["n_cells_fitted"],
        "years": fit["n_years_observed"],
        "dispersion": fit["dispersion"],
        "band_inflation": fit["band_inflation_from_overdispersion"],
        "year_df": fit["year_df"],
    })
quality = pl.DataFrame(rows)
print(quality)

# Population data is far from Poisson; insured data much closer; and controlling
# for duration on ILEC takes it closest of all.
d = {row["run"]: row["dispersion"] for row in rows}
assert d["HMD USA"] > d["HMD E&W"] > d["ILEC pooled"] > d["ILEC banded"]
assert d["ILEC banded"] < 1.25, "duration banding should bring phi near nominal"
print("\nphi ordering holds: population >> insured pooled > insured duration-controlled")

## 2. The slowdown, by age and population

The claim in `MEASUREMENT_experience_gam_hmd.md` is that the post-2010 slowdown is
**real and age-localised**, not uniform. A separable model would have averaged
these into one number and lost the sign change.

`resolvable` is the harness's `bands_overlap` inverted — and it is *indicative*,
not a significance test for the difference: the two window contrasts come from the
same fitted coefficients and are correlated.

In [ ]:
def windows(name: str) -> pl.DataFrame:
    wc = REPORTS[name]["window_comparison"]
    return pl.DataFrame(wc["rows"]).select(
        pl.lit(name).alias("run"),
        "attained_age",
        (pl.col("early_annualised_mi") * 100).round(2).alias("early_%"),
        (pl.col("late_annualised_mi") * 100).round(2).alias("late_%"),
        (pl.col("delta") * 100).round(2).alias("delta_pp"),
        (~pl.col("bands_overlap")).alias("resolvable"),
    )


hmd = pl.concat([windows("HMD USA"), windows("HMD E&W")])
print(hmd)

# Both populations: improvement at 45-65 collapses, resolvably, in both.
midlife = hmd.filter(pl.col("attained_age").is_in([45, 55, 65]))
assert (midlife["delta_pp"] < 0).all(), "midlife must slow in both populations"
assert midlife["resolvable"].all(), "and be resolvable in both"
print("\nmidlife collapse replicates independently in USA and E&W")

### The oldest ages diverge — and the fit's own baselines explain it

USA age 85 accelerates sharply; England & Wales does not move resolvably. The
1990s levels say why: US old-age mortality was *worsening* while E&W was already
improving, so the US had catch-up available.

In [ ]:
old = hmd.filter(pl.col("attained_age") == 85)
print(old)

usa_85 = old.filter(pl.col("run") == "HMD USA").row(0, named=True)
ew_85 = old.filter(pl.col("run") == "HMD E&W").row(0, named=True)

assert usa_85["early_%"] < 0 < ew_85["early_%"], "US was worsening at 85; E&W improving"
assert usa_85["delta_pp"] > ew_85["delta_pp"], "so the US had more room and took it"
assert usa_85["resolvable"] and not ew_85["resolvable"]
print(
    f"\nUSA 85: {usa_85['early_%']:+.2f}% -> {usa_85['late_%']:+.2f}%  (resolvable)\n"
    f"E&W 85: {ew_85['early_%']:+.2f}% -> {ew_85['late_%']:+.2f}%  (not resolvable)"
)

## 3. Quantifying the duration effect

This is the one the pooled ILEC fit got wrong. Duration mix drifting with calendar
year was being absorbed by the `te(age, calendar_year)` tensor and read as *less
improvement*.

The duration representative is calendar-invariant by construction, so the duration
term cancels exactly in the calendar contrast and cannot shift MI directly. What
changes is the **cell set** — which is why the surface legitimately moves.

In [ ]:
def late_window(name: str) -> pl.DataFrame:
    rows = REPORTS[name]["window_comparison"]["rows"]
    return pl.DataFrame(rows).select(
        "attained_age",
        (pl.col("late_annualised_mi") * 100).round(2).alias(name),
    )


duration_effect = late_window("ILEC pooled").join(
    late_window("ILEC banded"), on="attained_age"
).with_columns(
    (pl.col("ILEC banded") - pl.col("ILEC pooled")).round(2).alias("effect_pp")
)
print(duration_effect)

# Controlling for duration raises fitted improvement at every age but the
# boundary-contaminated youngest one.
assert (duration_effect.filter(pl.col("attained_age") >= 55)["effect_pp"] > 0).all()
worst = duration_effect["effect_pp"].abs().max()
print(f"\nlargest single-age duration effect: {worst:.2f} percentage points/yr")

## 4. Experience versus mix — the decomposition

Direct standardisation on a complete cell panel. `standardised_ae` is what A/E
would have done had the book's composition never changed, so its slope is the
**experience** signal; `mix_effect` is what composition alone contributed.

**A flat crude A/E is not evidence that assumptions are sound.** It can be two
effects of opposite sign, either of which can move independently.

In [ ]:
banded = REPORTS["ILEC banded"]
decomp = banded.get("standardised_ae")

if decomp is None:
    DEGRADED.append("standardised_ae")
    print(
        "No standardised_ae section in the committed report.\n"
        "It was added after this run; re-run the ILEC harness to populate it:\n"
        "  uv run python scripts/experience_diligence.py --source ilec \\\n"
        "      --year-df 3 --duration-bands -o ilec.json --markdown ilec.md"
    )
else:
    frame = pl.DataFrame(decomp["rows"])
    print(frame)
    print(
        f"\ncrude slope        {decomp['crude_slope_per_year']:+.5f} /yr\n"
        f"experience slope   {decomp['standardised_slope_per_year']:+.5f} /yr\n"
        f"mix slope          {decomp['mix_slope_per_year']:+.5f} /yr\n"
        f"panel coverage     {decomp['coverage_share'] * 100:.1f}% of expected deaths"
    )
    # The decomposition is additive by construction.
    np.testing.assert_allclose(
        decomp["standardised_slope_per_year"] + decomp["mix_slope_per_year"],
        decomp["crude_slope_per_year"],
        rtol=1e-6,
        atol=1e-9,
    )
    print("\nadditive: experience + mix == crude")

    # The measured verdict on MEASUREMENT_experience_gam_ilec.md §4. The
    # DIRECTION of both components matches the inference that preceded them:
    # experience improving faster than SOA assumed, mix pushing the other way.
    assert decomp["standardised_slope_per_year"] < 0, "experience must drift down"
    assert decomp["mix_slope_per_year"] > 0, "mix must push the other way"

    # The MAGNITUDE is what the inference got wrong. Mix offsets only about a
    # fifth of the experience signal — it is not a co-equal cancellation, which
    # is what "two offsetting effects" implied before this was measured.
    share = abs(decomp["mix_slope_per_year"]) / abs(decomp["standardised_slope_per_year"])
    assert share < 0.35, "mix is a modest offset, not a co-equal one"
    print(f"mix offsets {share * 100:.1f}% of the experience signal")

    # And the mix series is noisy: sign changes across an 8-point series mean the
    # slope is weak evidence for a steady drift, whatever its sign.
    mix = [row["mix_effect"] for row in decomp["rows"]]
    flips = sum(1 for a, b in zip(mix, mix[1:]) if a * b < 0)
    assert flips >= 2, "if the series ever becomes monotone, revisit the caveat"
    print(f"mix series changes sign {flips} times — the slope is weak evidence")

## 5. Insured versus population

PLAN §2 named this as the interesting output: insured lives are underwritten, so a
model showing them identical to the general population would have a bug. The
*shape* of the difference is the finding.

Windows are not identical — HMD 2010–2019 against ILEC 2016–2019 — so this
compares overlapping regimes rather than the same years. Age 45 is excluded: it is
still boundary-contaminated on the ILEC fit.

In [ ]:
pop = late_window("HMD USA").rename({"HMD USA": "population_%"})
ins = late_window("ILEC banded").rename({"ILEC banded": "insured_%"})
divergence = (
    pop.join(ins, on="attained_age")
    .filter(pl.col("attained_age") >= 55)
    .with_columns((pl.col("insured_%") - pl.col("population_%")).round(2).alias("gap_pp"))
)
print(divergence)

# Insured improve faster where underwriting screens out the midlife stagnation,
# and converge (or fall behind) once selection has worn off.
midlife_gap = divergence.filter(pl.col("attained_age").is_in([55, 65]))["gap_pp"]
assert (midlife_gap > 0.5).all(), "insured must outpace population at 55-65"
oldest = divergence.filter(pl.col("attained_age") == 85).row(0, named=True)
assert oldest["gap_pp"] < midlife_gap.min(), "and the gap must close by 85"
print("\ninsured outpace the population at midlife; the gap closes by 85")

## 6. What this notebook does not show

- **No plots.** Numbers and tables diff in git; images do not. The measurement
  documents and this notebook are both designed to be reviewed as text.
- **No licensed data.** Every cell above reads committed aggregates. Reproducing
  the *fits* needs the HMD and SOA-ILEC files and
  `docs/RUNBOOK_experience_data_acquisition.md`.
- **No significance test on the window differences.** The two contrasts share
  fitted coefficients; `resolvable` is indicative only.
- **The mix decomposition covers only the complete panel.** Check
  `coverage_share` before reading its slope as the whole book.

## 7. Sections that could not run

A committed report can predate a harness feature the notebook reads. That is
deliberate — notebook and harness evolve at different rates — but a section
that quietly prints instead of asserting would overstate this notebook's
coverage claim. So the gaps are collected and pinned by the test suite.

In [ ]:
if DEGRADED:
    print("sections not verified against committed reports:")
    for name in DEGRADED:
        print(f"  - {name}")
else:
    print("every section ran against a committed report")
